# Tarea Práctica: Calidad de Datos en IoT (Smart Building)

## Contexto
Estás trabajando como Data Engineer para una empresa de edificios inteligentes.
Tenéis una red de sensores de temperatura y humedad desplegada en las oficinas.
Sin embargo, la red es inestable: a veces los sensores pierden conexión, envían datos duplicados o se descalibran y mandan valores imposibles.

## Objetivo
Tu misión es limpiar el dataset raw (sucio) para que pueda ser usado por el equipo de Data Science para predecir el consumo energético.

Debes resolver los siguientes problemas:
1.  **Completitud:** Gestionar los datos faltantes.
2.  **Consistencia:** Eliminar duplicados.
3.  **Precisión:** Filtrar valores erróneos (Outliers).

### !IMPORATANTE, SI TOMAS UNA DECISIÓN , QUIERO QUE ME LA JUSTIFIQUES. NOTA: SI HAY IA, TENDRAS UN PROBLEMA, You know what I mean.!

In [6]:
import pandas as pd
import numpy as np

# --- CÓDIGO DE GENERACIÓN DE DATOS (NO MODIFICAR) ---

data = {
    'sensor_id': ['S-01', 'S-02', 'S-01', 'S-03', 'S-04', 'S-01', 'S-05', 'S-02'],
    'timestamp': [
        '2024-02-10 09:00', '2024-02-10 09:00',
        '2024-02-10 09:15', # S-01 lectura correcta
        '2024-02-10 09:15',
        '2024-02-10 09:15',
        '2024-02-10 09:00', # S-01 DUPLICADO (mismo timestamp que el primero)
        None,               # S-05 Timestamp perdido
        '2024-02-10 09:30'
    ],
    'temperatura': [22.5, 21.0, None, 23.5, 20.0, 22.5, 19.5, 999.0], # 999.0 es un outlier brutal
    'humedad': [45, 50, 44, 48, None, 45, 52, -100] # -100 humedad imposible
}

df_iot = pd.DataFrame(data)
print("--- Dataset Raw (Sucio) ---")
display(df_iot)

--- Dataset Raw (Sucio) ---


,sensor_id,timestamp,temperatura,humedad
0,S-01,2024-02-10 09:00,22.5,45.0
1,S-02,2024-02-10 09:00,21.0,50.0
2,S-01,2024-02-10 09:15,NaN,44.0
3,S-03,2024-02-10 09:15,23.5,48.0
4,S-04,2024-02-10 09:15,20.0,NaN
5,S-01,2024-02-10 09:00,22.5,45.0
6,S-05,None,19.5,52.0
7,S-02,2024-02-10 09:30,999.0,-100.0


---

### Ejercicio 1: Completitud
**Problema:** Hay sensores que han fallado y no han enviado temperatura o humedad (`None`/`NaN`). También hay un timestamp perdido.

**Tarea:**
1.  Identifica cuántos valores nulos hay en cada columna.
2.  Elimina las filas que no tengan `timestamp` (es crítico).
3.  Para `temperatura` y `humedad`, decide si borrar o rellenar (puedes usar la media o un valor fijo).

### Justificacion Teorica - Ejercicio 1: Completitud

**Concepto aplicado (UT3):** La **completitud** responde a la pregunta basica: *tenemos todo lo necesario?* Segun el dossier, puede faltar un campo, pueden faltar filas, o puede faltar un tramo temporal entero. En redes IoT, la incompletitud aparece tipicamente por saturacion en la ingesta, time-outs o caidas momentaneas de los nodos sensores.

**Por que eliminar la fila sin timestamp:** El `timestamp` es la clave temporal del evento. Sin el, el dato no tiene posicion en la serie temporal y no puede correlacionarse con otros sensores ni sistemas. La fila de S-05 es inutilizable para el objetivo de prediccion energetica: no sabemos cuando ocurrio.

**Por que imputar temperatura y humedad con la media del sensor:** El evento de S-01 (09:15) y S-04 (09:15) tienen timestamp valido, asi que sabemos *cuando* ocurrieron. En lugar de eliminar la fila y perder ese punto temporal, imputamos con la **media agrupada por `sensor_id`** (no la global), porque cada sensor esta ubicado en una zona del edificio con condiciones termicas distintas. Usar la media global mezclaria zonas incomparables, introduciendo un sesgo. La imputacion por sensor es la estrategia mas adecuada al uso: series temporales continuas para modelos de ML.

In [7]:
# =============================================================
# EJERCICIO 1: COMPLETITUD
# =============================================================
# En una red de sensores IoT es habitual que los dispositivos
# pierdan conectividad y no transmitan todos sus campos.
# Segun la teoria de calidad de datos (UT3), la COMPLETITUD mide
# el grado en que los datos requeridos estan presentes.
# Debemos decidir que hacer con cada tipo de nulo:
#   - timestamp nulo: CRITICO. Sin referencia temporal no podemos
#     ordenar ni correlacionar eventos. La fila es inutilizable.
#   - temperatura/humedad nulos: imputar con la media del sensor
#     si tiene otras lecturas; si no, usar la media global.

# Paso 1: Auditoria de nulos (inspeccion antes de actuar)
print("=== AUDITORIA DE VALORES NULOS ===")
print(df_iot.isnull().sum())
print()
print("Porcentaje de nulos por columna:")
print((df_iot.isnull().sum() / len(df_iot) * 100).round(2))

# Paso 2: Eliminar filas sin timestamp
# Justificacion: el timestamp es la clave temporal del evento.
# Sin el no se puede situar el dato en la serie temporal ni
# hacer joins con otros sistemas. Es un campo critico (PK parcial).
df_sin_nulos_ts = df_iot.dropna(subset=['timestamp'])
print(f"\nFilas eliminadas por timestamp nulo: {len(df_iot) - len(df_sin_nulos_ts)}")

# Paso 3: Convertir timestamp a datetime para operaciones futuras
df_sin_nulos_ts = df_sin_nulos_ts.copy()
df_sin_nulos_ts['timestamp'] = pd.to_datetime(df_sin_nulos_ts['timestamp'])

# Paso 4: Imputar temperatura con la media del sensor (1er intento)
# Si el sensor no tiene suficientes lecturas validas, la media del
# grupo es NaN -> usamos fillna con la media global como fallback.
df_sin_nulos_ts['temperatura'] = df_sin_nulos_ts.groupby('sensor_id')['temperatura']\
    .transform(lambda x: x.fillna(x.mean()))
# Fallback: si sigue habiendo NaN (sensor sin ninguna lectura valida),
# imputar con la media global de temperatura del edificio.
df_sin_nulos_ts['temperatura'] = df_sin_nulos_ts['temperatura'].fillna(
    df_sin_nulos_ts['temperatura'].mean()
)

# Paso 5: Imputar humedad con la media del sensor + fallback global
# Mismo razonamiento: S-04 no tiene otra lectura de humedad,
# su media de grupo es NaN -> fallback a media global del edificio.
df_sin_nulos_ts['humedad'] = df_sin_nulos_ts.groupby('sensor_id')['humedad']\
    .transform(lambda x: x.fillna(x.mean()))
df_sin_nulos_ts['humedad'] = df_sin_nulos_ts['humedad'].fillna(
    df_sin_nulos_ts['humedad'].mean()
)

print("\n=== DATASET TRAS GESTIONAR COMPLETITUD ===")
display(df_sin_nulos_ts)
print(f"Nulos restantes: {df_sin_nulos_ts.isnull().sum().sum()}")

# Trabajamos con este DataFrame de aqui en adelante
df_clean = df_sin_nulos_ts.copy()

=== AUDITORIA DE VALORES NULOS ===
sensor_id      0
timestamp      1
temperatura    1
humedad        1
dtype: int64

Porcentaje de nulos por columna:
sensor_id       0.0
timestamp      12.5
temperatura    12.5
humedad        12.5
dtype: float64

Filas eliminadas por timestamp nulo: 1

=== DATASET TRAS GESTIONAR COMPLETITUD ===


,sensor_id,timestamp,temperatura,humedad
0,S-01,2024-02-10 09:00:00,22.5,45.0
1,S-02,2024-02-10 09:00:00,21.0,50.0
2,S-01,2024-02-10 09:15:00,22.5,44.0
3,S-03,2024-02-10 09:15:00,23.5,48.0
4,S-04,2024-02-10 09:15:00,20.0,22.0
5,S-01,2024-02-10 09:00:00,22.5,45.0
7,S-02,2024-02-10 09:30:00,999.0,-100.0


Nulos restantes: 0


---

### Ejercicio 2: Consistencia
**Problema:** La red a veces reenvía paquetes. El sensor `S-01` parece tener una lectura duplicada a las `09:00`.

**Tarea:**
1.  Detecta los duplicados basándote en `sensor_id` y `timestamp`.
2.  Elimínalos manteniendo solo la primera ocurrencia.

### Justificacion Teorica - Ejercicio 2: Consistencia

**Concepto aplicado (UT3):** La **consistencia** responde a la pregunta: los datos encajan entre si, o cuentan historias incompatibles? Un problema clasico de inconsistencia en sistemas distribuidos son los **duplicados por reintentos** (duplicate replay). Tal como establece el dossier: para no perder datos, se reintenta; pero reintentar tiene un precio: duplicar.

**Por que aparecen duplicados en IoT:** La red usa garantias de entrega at-least-once: el broker o el propio sensor reenvian un paquete si no reciben confirmacion (ACK) del consumidor. Esto es preferible a perder el evento, pero genera copias exactas identificables por la combinacion sensor_id + timestamp (clave natural del evento). Sin un mecanismo de idempotencia aguas abajo, los duplicados entrarian al pipeline y distorsionarian cualquier agregado: medias de temperatura, consumo energetico calculado, etc.

**Por que usar keep=first:** La primera ocurrencia es el evento original; la segunda es el reenvio de la red. Conservar la primera y descartar las siguientes es la estrategia estandar de deduplicacion. El subset=['sensor_id', 'timestamp'] actua como clave natural y garantiza que solo se comparan eventos del mismo sensor en el mismo instante, sin afectar a lecturas legitimas de otros sensores o momentos distintos.

In [8]:
# EJERCICIO 2: CONSISTENCIA
# En redes IoT es frecuente el reenvio de paquetes (QoS at-least-once):
# el broker o el propio sensor puede reenviar un mensaje si no recibe ACK.
# Esto genera duplicados exactos en sensor_id + timestamp.
# La estrategia correcta es deduplicar manteniendo la primera ocurrencia,
# ya que el evento original es el valido y el duplicado es solo un reenvio.

# Paso 1: Detectar duplicados por sensor_id y timestamp
duplicados = df_clean[df_clean.duplicated(subset=['sensor_id', 'timestamp'], keep='first')]
print("=== FILAS DUPLICADAS DETECTADAS ===")
print(f"Numero de duplicados: {len(duplicados)}")
display(duplicados)

# Paso 2: Eliminar duplicados manteniendo la primera ocurrencia
# keep='first' asegura que nos quedamos con el evento original.
df_clean = df_clean.drop_duplicates(subset=['sensor_id', 'timestamp'], keep='first')

print(f"\nFilas antes de deduplicar: {len(df_sin_nulos_ts)}")
print(f"Filas despues de deduplicar: {len(df_clean)}")
print("\n=== DATASET TRAS ELIMINAR DUPLICADOS ===")
display(df_clean)

=== FILAS DUPLICADAS DETECTADAS ===
Numero de duplicados: 1


,sensor_id,timestamp,temperatura,humedad
5,S-01,2024-02-10 09:00:00,22.5,45.0



Filas antes de deduplicar: 7
Filas despues de deduplicar: 6

=== DATASET TRAS ELIMINAR DUPLICADOS ===


,sensor_id,timestamp,temperatura,humedad
0,S-01,2024-02-10 09:00:00,22.5,45.0
1,S-02,2024-02-10 09:00:00,21.0,50.0
2,S-01,2024-02-10 09:15:00,22.5,44.0
3,S-03,2024-02-10 09:15:00,23.5,48.0
4,S-04,2024-02-10 09:15:00,20.0,22.0
7,S-02,2024-02-10 09:30:00,999.0,-100.0


---

### Ejercicio 3: Precisión
**Problema:**
*   El sensor `S-02` ha marcado **999.0°C**. ¡El edificio estaría en llamas!
*   La humedad de **-100%** es físicamente imposible.

**Tarea:**
1.  Filtra el DataFrame para eliminar (o corregir) estos valores imposibles.
    *   Temperatura válida: entre -10 y 50 ºC
    *   Humedad válida: entre 0 y 100 %


### Justificacion Teorica - Ejercicio 3: Precision

**Concepto aplicado (UT3):** La **precision** responde a la pregunta: representa este valor la realidad? Un dato puede estar bien formateado y aun asi ser incorrecto. El dossier lo ejemplifica con sensores: si un sensor registra 80 C cuando la maquina esta a 40 C, el dato es impreciso aunque numericamente sea valido.

**Por que aparecen valores imposibles en IoT:** Los sensores pueden descalibrarse o sufrir fallos de hardware (*out-of-range readings*). En sistemas distribuidos, estos valores entran silenciosamente al pipeline porque el formato del mensaje es correcto (JSON valido, tipo float), pero el contenido viola las leyes fisicas del dominio. El dossier denomina esto **fallo silencioso**: el pipeline termina bien, pero produce datos invalidos.

**Por que usar cuarentena en lugar de descarte silencioso:** Segun el dossier, cuando el sistema detecta algo raro debe tener una salida controlada. La cuarentena (*quarantine / dead-letter*) separa el dato problematico, lo guarda con el motivo del fallo, y lo deja disponible para auditoria. Asi el equipo de hardware puede ser notificado sobre el sensor S-02 descalibrado. Un descarte silencioso ocultaria el problema sistematico.

**Por que estos rangos ([-10, 50] C y [0, 100] %):** Son los limites fisicamente plausibles para un edificio de oficinas en cualquier condicion operacional. 999 C y -100 % son valores termica y fisicamente imposibles que solo pueden explicarse por un sensor roto, nunca por una condicion real del edificio.

In [9]:
# EJERCICIO 3: PRECISION
# Un sensor descalibrado o con fallo de hardware puede emitir valores
# fisicamente imposibles. En IoT esto se conoce como 'noisy data' o
# 'out-of-range readings'. Segun los limites fisicos del edificio:
#   - Temperatura: entre -10 y 50 C (fuera de este rango es imposible
#     en un edificio de oficinas bajo cualquier condicion operacional)
#   - Humedad: entre 0% y 100% (la humedad relativa no puede ser negativa
#     ni superar el 100% por definicion fisica)
# Mandamos a cuarentena las filas invalidas en lugar de simplemente
# descartarlas, para poder auditarlas y notificar al equipo de hardware.

# Definir rangos validos segun enunciado
TEMP_MIN, TEMP_MAX = -10, 50
HUM_MIN, HUM_MAX = 0, 100

# Mascara booleana de filas validas
mascara_valida = (
    (df_clean['temperatura'] >= TEMP_MIN) & (df_clean['temperatura'] <= TEMP_MAX) &
    (df_clean['humedad'] >= HUM_MIN) & (df_clean['humedad'] <= HUM_MAX)
)

# Filas que van a cuarentena (outliers / valores imposibles)
df_cuarentena = df_clean[~mascara_valida].copy()
print("=== FILAS ENVIADAS A CUARENTENA (VALORES IMPOSIBLES) ===")
print(f"Registros en cuarentena: {len(df_cuarentena)}")
display(df_cuarentena)

# Dataset limpio: solo valores dentro de rangos fisicamente validos
df_clean = df_clean[mascara_valida].copy()

print(f"\nFilas eliminadas por precision: {len(df_cuarentena)}")
print(f"Filas validas restantes: {len(df_clean)}")
print("\n=== DATASET TRAS FILTRAR OUTLIERS ===")
display(df_clean)

=== FILAS ENVIADAS A CUARENTENA (VALORES IMPOSIBLES) ===
Registros en cuarentena: 1


,sensor_id,timestamp,temperatura,humedad
7,S-02,2024-02-10 09:30:00,999.0,-100.0



Filas eliminadas por precision: 1
Filas validas restantes: 5

=== DATASET TRAS FILTRAR OUTLIERS ===


,sensor_id,timestamp,temperatura,humedad
0,S-01,2024-02-10 09:00:00,22.5,45.0
1,S-02,2024-02-10 09:00:00,21.0,50.0
2,S-01,2024-02-10 09:15:00,22.5,44.0
3,S-03,2024-02-10 09:15:00,23.5,48.0
4,S-04,2024-02-10 09:15:00,20.0,22.0


---

### Resultado Final
Muestra cómo ha quedado el DataFrame limpio y cuántas filas tiene ahora.

In [5]:
# RESULTADO FINAL
# Resumen de todas las transformaciones aplicadas al pipeline de limpieza:
# 1. Completitud: eliminada 1 fila sin timestamp (S-05), imputados NaN de
#    temperatura (S-01 a las 09:15) y humedad (S-04) con media del sensor.
# 2. Consistencia: eliminado 1 duplicado (S-01 a las 09:00, reenvio de red).
# 3. Precision: 1 fila a cuarentena (S-02 con 999 C y -100% humedad).
# El dataset resultante es integro, completo y fisicamente valido.

print("======================================================")
print(" PIPELINE DE CALIDAD COMPLETADO - RESUMEN FINAL")
print("======================================================")
print(f"Filas originales (dataset raw):    {len(df_iot)}")
print(f"Filas tras gestionar completitud:  {len(df_sin_nulos_ts)}")
print(f"Filas tras eliminar duplicados:    {len(df_sin_nulos_ts.drop_duplicates(subset=['sensor_id','timestamp']))}")
print(f"Filas finales limpias:             {len(df_clean)}")
print(f"Filas en cuarentena (outliers):    {len(df_cuarentena)}")
print("")
print("=== DATASET FINAL LIMPIO ===")
display(df_clean)
print("")
print("Tipos de datos finales:")
print(df_clean.dtypes)
print("")
print("Estadisticas descriptivas del dataset limpio:")
display(df_clean.describe())

 PIPELINE DE CALIDAD COMPLETADO - RESUMEN FINAL
Filas originales (dataset raw):    8
Filas tras gestionar completitud:  7
Filas tras eliminar duplicados:    6
Filas finales limpias:             5
Filas en cuarentena (outliers):    1

=== DATASET FINAL LIMPIO ===


,sensor_id,timestamp,temperatura,humedad
0,S-01,2024-02-10 09:00:00,22.5,45.0
1,S-02,2024-02-10 09:00:00,21.0,50.0
2,S-01,2024-02-10 09:15:00,22.5,44.0
3,S-03,2024-02-10 09:15:00,23.5,48.0
4,S-04,2024-02-10 09:15:00,20.0,22.0



Tipos de datos finales:
sensor_id              object
timestamp      datetime64[ns]
temperatura           float64
humedad               float64
dtype: object

Estadisticas descriptivas del dataset limpio:


,timestamp,temperatura,humedad
count,5,5.000000,5.000000
mean,2024-02-10 09:09:00,21.900000,41.800000
min,2024-02-10 09:00:00,20.000000,22.000000
25%,2024-02-10 09:00:00,21.000000,44.000000
50%,2024-02-10 09:15:00,22.500000,45.000000
75%,2024-02-10 09:15:00,22.500000,48.000000
max,2024-02-10 09:15:00,23.500000,50.000000
std,NaN,1.387444,11.322544
